In [ ]:
import sys
sys.path.append("..")
import pandas as pd
from src.variants import load_missense, validate_against_structure
from src.structure import per_residue_table

acc = "P01116"
df = validate_against_structure(load_missense(acc), acc)
res = per_residue_table(acc)
res["buried"] = res["rsa"] < 0.2          # common cutoff for "buried"
print(res["ss"].value_counts())
print(f"{res['buried'].mean():.0%} of residues are buried")
merged = df.merge(res, on="position")
merged.head()

In [ ]:
sub = merged[merged["label"].isin(["pathogenic", "benign"])]
print(pd.crosstab(sub["label"], sub["ss"], normalize="index").round(2))
print(sub.groupby("label")[["rsa", "plddt"]].median())

In [ ]:
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
import seaborn as sns

p = sub[sub.label == "pathogenic"]
b = sub[sub.label == "benign"]
for col in ["rsa", "plddt"]:
    stat, pval = mannwhitneyu(p[col], b[col])
    print(f"{col}: Mann-Whitney p = {pval:.4g}")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(data=sub, x="label", y="rsa", ax=ax[0])
sns.boxplot(data=sub, x="label", y="plddt", ax=ax[1])
plt.tight_layout()
plt.savefig("../results/figures/pathogenic_vs_benign.png", dpi=200)

In [ ]:
res["ss"].value_counts(normalize=True).round(2)   # background
merged[merged.label == "pathogenic"]["ss"].value_counts(normalize=True).round(2)